# Сезонная модель коррекции CatBoost

Основная модель `depth=6` прогнозирует GMV по текущим 216 признакам. Этот ноутбук проверяет, можно ли исправить её зимний OOF-прогноз с помощью отдельного неглубокого CatBoost.

Корректор предсказывает остаток в пространстве `log1p`:

`residual = log1p(target) - log1p(base_prediction)`.

Сравниваются неизменённый прогноз, постоянная поправка, корректор по текущим признакам и такой же корректор с year-over-year признаками.

## 0. Режим запуска

`RUN_CV=True` запускает пять cross-user фолдов для двух моделей. `TRAIN_FULL_CORRECTORS=True` обучает диагностические корректоры на всех пользователях зимнего якоря и сохраняет модели. После выполнения оба флага выключаются.

In [1]:
RUN_CV = False
TRAIN_FULL_CORRECTORS = False

VALIDATION_ANCHOR = '2026-01-14'
N_USER_FOLDS = 5
CORRECTION_LIMIT = 0.20

## 1. Импорты и артефакты

In [2]:
from __future__ import annotations

from datetime import date
import gc
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    CATBOOST_CORRECTION_DIR,
    CATBOOST_SEASONALITY_DIR,
    CATBOOST_TUNING_DIR,
    CATBOOST_V2_SNAPSHOT_DIR,
    ensure_output_dirs,
)
from src.correction import (
    apply_log_correction,
    correction_target,
    make_correction_model,
)
from src.validation import feature_columns, rmsle

ensure_output_dirs()
anchor = date.fromisoformat(VALIDATION_ANCHOR)

## 2. Сбор зимней матрицы

Используется честный OOF-прогноз основной модели для якоря `2026-01-14`. Текущие признаки построены только по данным не позже якоря, сезонные — по соответствующему периоду 2025 года. `user_id` служит только для соединения и формирования фолда.

In [3]:
base_oof = pd.read_parquet(CATBOOST_TUNING_DIR / 'depth_winner_oof.parquet')
base_oof['validation_anchor'] = pd.to_datetime(base_oof['validation_anchor']).dt.date
base_oof = base_oof[base_oof['validation_anchor'] == anchor].copy()

current_snapshot = pl.read_parquet(
    CATBOOST_V2_SNAPSHOT_DIR / f'train_{VALIDATION_ANCHOR}.parquet'
)
current_features = feature_columns(current_snapshot)
assert len(current_features) == 216
current_pd = current_snapshot.select(['user_id', *current_features]).to_pandas()

seasonal_snapshot = pl.read_parquet(
    CATBOOST_SEASONALITY_DIR / f'features_{VALIDATION_ANCHOR}.parquet'
)
seasonal_features = [
    column for column in seasonal_snapshot.columns
    if column not in {'user_id', 'anchor_date'}
]
seasonal_pd = seasonal_snapshot.select(['user_id', *seasonal_features]).to_pandas()

dataset = (
    base_oof[['user_id', 'target', 'prediction']]
    .merge(current_pd, on='user_id', how='left', validate='one_to_one')
    .merge(seasonal_pd, on='user_id', how='left', validate='one_to_one')
)
dataset['base_log_prediction'] = np.log1p(np.clip(dataset['prediction'], 0, None))
dataset['correction_target'] = correction_target(
    dataset['target'].to_numpy(), dataset['prediction'].to_numpy()
)
dataset['user_fold'] = (dataset['user_id'].to_numpy() % N_USER_FOLDS).astype(np.int8)

assert len(dataset) == 250_000
assert not dataset.isna().any().any()
print(f'Строк: {len(dataset):,}')
print(f'Текущих признаков: {len(current_features)}')
print(f'Сезонных признаков: {len(seasonal_features)}')
print(f'Базовый RMSLE: {rmsle(dataset.target.to_numpy(), dataset.prediction.to_numpy()):.6f}')

Строк: 250,000
Текущих признаков: 216
Сезонных признаков: 59
Базовый RMSLE: 1.703598


## 3. Наборы признаков

Контрольный корректор видит базовый прогноз и все текущие v2-признаки. Сезонный корректор отличается только добавлением year-over-year колонок. Поэтому разность их RMSLE показывает дополнительную ценность прошлого года.

In [4]:
control_feature_columns = ['base_log_prediction', *current_features]
seasonal_feature_columns = [*control_feature_columns, *seasonal_features]

feature_sets = {
    'current_correction': control_feature_columns,
    'seasonal_correction': seasonal_feature_columns,
}
pd.DataFrame([
    {'model': name, 'n_features': len(columns)}
    for name, columns in feature_sets.items()
])

,model,n_features
0,current_correction,217
1,seasonal_correction,276


## 4. Cross-user OOF

Пользователи делятся детерминированно по `user_id % 5`. На каждом шаге корректор обучается на 80% пользователей и прогнозирует остальные 20%. Поправка ограничивается диапазоном `[-0.2, +0.2]` в `log1p`, что соответствует приблизительно изменению GMV не более чем на 18–22%.

In [5]:
def run_user_fold_correction(label, columns):
    raw_correction = np.zeros(len(dataset), dtype=np.float32)
    corrected_prediction = np.zeros(len(dataset), dtype=np.float32)
    fold_rows = []

    for fold in range(N_USER_FOLDS):
        print(f'[{label}] fold {fold}')
        train_mask = dataset['user_fold'].to_numpy() != fold
        valid_mask = ~train_mask
        model = make_correction_model(random_seed=42 + fold)
        model.fit(
            dataset.loc[train_mask, columns],
            dataset.loc[train_mask, 'correction_target'],
        )
        fold_raw = model.predict(dataset.loc[valid_mask, columns])
        fold_prediction = apply_log_correction(
            dataset.loc[valid_mask, 'prediction'].to_numpy(),
            fold_raw,
            correction_limit=CORRECTION_LIMIT,
        )
        raw_correction[valid_mask] = fold_raw
        corrected_prediction[valid_mask] = fold_prediction
        fold_rows.append({
            'model': label,
            'fold': fold,
            'users': int(valid_mask.sum()),
            'base_rmsle': rmsle(
                dataset.loc[valid_mask, 'target'].to_numpy(),
                dataset.loc[valid_mask, 'prediction'].to_numpy(),
            ),
            'corrected_rmsle': rmsle(
                dataset.loc[valid_mask, 'target'].to_numpy(),
                fold_prediction,
            ),
            'mean_raw_correction': float(np.mean(fold_raw)),
            'clip_share': float(np.mean(np.abs(fold_raw) > CORRECTION_LIMIT)),
        })
        del model
        gc.collect()

    return raw_correction, corrected_prediction, pd.DataFrame(fold_rows)

cv_predictions_path = CATBOOST_CORRECTION_DIR / 'cv_predictions.parquet'
cv_fold_metrics_path = CATBOOST_CORRECTION_DIR / 'fold_metrics.csv'

if RUN_CV:
    prediction_output = dataset[['user_id', 'target', 'prediction', 'user_fold']].copy()
    fold_metric_parts = []
    for label, columns in feature_sets.items():
        raw, corrected, fold_metrics = run_user_fold_correction(label, columns)
        prediction_output[f'{label}_raw'] = raw
        prediction_output[f'{label}_prediction'] = corrected
        fold_metric_parts.append(fold_metrics)
    fold_metrics = pd.concat(fold_metric_parts, ignore_index=True)
    prediction_output.to_parquet(cv_predictions_path, index=False)
    fold_metrics.to_csv(cv_fold_metrics_path, index=False)
else:
    if not cv_predictions_path.exists() or not cv_fold_metrics_path.exists():
        raise FileNotFoundError('Нет CV-результатов. Установите RUN_CV=True.')
    prediction_output = pd.read_parquet(cv_predictions_path)
    fold_metrics = pd.read_csv(cv_fold_metrics_path)

display(fold_metrics)

,model,fold,users,base_rmsle,corrected_rmsle,mean_raw_correction,clip_share
0,current_correction,0,50300,1.703362,1.683853,-0.259338,0.708549
1,current_correction,1,49925,1.704430,1.684643,-0.258979,0.693801
2,current_correction,2,49662,1.705696,1.686229,-0.259473,0.701321
3,current_correction,3,49921,1.698283,1.680052,-0.262844,0.713507
4,current_correction,4,50192,1.706205,1.686928,-0.260368,0.704933
5,seasonal_correction,0,50300,1.703362,1.681544,-0.259444,0.632684
6,seasonal_correction,1,49925,1.704430,1.682654,-0.259012,0.643966
7,seasonal_correction,2,49662,1.705696,1.683797,-0.259161,0.629717
8,seasonal_correction,3,49921,1.698283,1.678117,-0.262362,0.640692
9,seasonal_correction,4,50192,1.706205,1.684791,-0.260987,0.639445


## 5. Постоянная поправка

Для каждого user-fold средний residual оценивается на остальных пользователях. Это простейшая честная калибровка, с которой должны сравниваться обучаемые корректоры.

In [6]:
constant_prediction = np.zeros(len(dataset), dtype=np.float32)
constant_rows = []
for fold in range(N_USER_FOLDS):
    train_mask = dataset['user_fold'].to_numpy() != fold
    valid_mask = ~train_mask
    mean_residual = float(dataset.loc[train_mask, 'correction_target'].mean())
    fold_prediction = apply_log_correction(
        dataset.loc[valid_mask, 'prediction'].to_numpy(),
        np.full(valid_mask.sum(), mean_residual),
        correction_limit=CORRECTION_LIMIT,
    )
    constant_prediction[valid_mask] = fold_prediction
    constant_rows.append({
        'fold': fold,
        'mean_residual': mean_residual,
        'corrected_rmsle': rmsle(
            dataset.loc[valid_mask, 'target'].to_numpy(), fold_prediction
        ),
    })
display(pd.DataFrame(constant_rows))

,fold,mean_residual,corrected_rmsle
0,0,-0.259873,1.684247
1,1,-0.259009,1.684956
2,2,-0.259595,1.686482
3,3,-0.262645,1.680468
4,4,-0.259843,1.687126


## 6. Итоговое сравнение

Сезонность считается полезной только если seasonal correction лучше одновременно неизменённого CatBoost, постоянной поправки и current correction.

In [7]:
comparison_rows = [
    {
        'model': 'base_catboost',
        'rmsle': rmsle(dataset.target.to_numpy(), dataset.prediction.to_numpy()),
    },
    {
        'model': 'constant_correction',
        'rmsle': rmsle(dataset.target.to_numpy(), constant_prediction),
    },
]
for label in feature_sets:
    comparison_rows.append({
        'model': label,
        'rmsle': rmsle(
            prediction_output.target.to_numpy(),
            prediction_output[f'{label}_prediction'].to_numpy(),
        ),
    })
comparison = pd.DataFrame(comparison_rows).sort_values('rmsle')
base_score = float(comparison.loc[comparison.model == 'base_catboost', 'rmsle'].iloc[0])
comparison['improvement_vs_base'] = base_score - comparison['rmsle']
current_score = float(comparison.loc[comparison.model == 'current_correction', 'rmsle'].iloc[0])
seasonal_score = float(comparison.loc[comparison.model == 'seasonal_correction', 'rmsle'].iloc[0])
seasonal_increment = current_score - seasonal_score
comparison.to_csv(CATBOOST_CORRECTION_DIR / 'comparison.csv', index=False)
display(comparison)
print(f'Дополнительное улучшение сезонности против current correction: {seasonal_increment:+.6f}')

,model,rmsle,improvement_vs_base
3,seasonal_correction,1.682182,0.021415
2,current_correction,1.684343,0.019255
1,constant_correction,1.684658,0.018940
0,base_catboost,1.703598,0.000000


Дополнительное улучшение сезонности против current correction: +0.002160


## 7. Стабильность по user-fold

Глобальная метрика может скрыть нестабильность. Сравниваем два корректора внутри каждой отложенной группы.

In [8]:
fold_pivot = fold_metrics.pivot(
    index='fold', columns='model', values='corrected_rmsle'
).reset_index()
fold_pivot['seasonal_improvement_vs_current'] = (
    fold_pivot['current_correction'] - fold_pivot['seasonal_correction']
)
display(fold_pivot)
print('Фолдов с выигрышем сезонности:', int((fold_pivot.seasonal_improvement_vs_current > 0).sum()), 'из', N_USER_FOLDS)

model,fold,current_correction,seasonal_correction,seasonal_improvement_vs_current
0,0,1.683853,1.681544,0.002309
1,1,1.684643,1.682654,0.001989
2,2,1.686229,1.683797,0.002433
3,3,1.680052,1.678117,0.001935
4,4,1.686928,1.684791,0.002136


Фолдов с выигрышем сезонности: 5 из 5


## 8. Обучение полных диагностических моделей

Обе модели обучаются на всех 250 тысячах пользователей только после cross-user оценки. Они сохраняются для анализа важности и потенциального применения к финальному базовому прогнозу. Сам submission здесь не создаётся.

In [9]:
importance_frames = []
for label, columns in feature_sets.items():
    model_path = CATBOOST_CORRECTION_DIR / f'{label}.cbm'
    importance_path = CATBOOST_CORRECTION_DIR / f'{label}_importance.csv'
    if TRAIN_FULL_CORRECTORS:
        print(f'Обучаем полную модель: {label}')
        model = make_correction_model(random_seed=42)
        model.fit(dataset[columns], dataset['correction_target'])
        model.save_model(str(model_path))
        importance = pd.DataFrame({
            'feature': columns,
            'importance': model.get_feature_importance(),
        }).sort_values('importance', ascending=False)
        importance.to_csv(importance_path, index=False)
        del model
        gc.collect()
    else:
        if not importance_path.exists():
            raise FileNotFoundError(f'Нет {importance_path.name}.')
        importance = pd.read_csv(importance_path)
    importance['model'] = label
    importance_frames.append(importance)

all_importance = pd.concat(importance_frames, ignore_index=True)
seasonal_importance = (
    all_importance[
        (all_importance.model == 'seasonal_correction')
        & (all_importance.feature.isin(seasonal_features))
    ]
    .sort_values('importance', ascending=False)
)
display(seasonal_importance.head(20))

,feature,importance,model
217,ly_weekday_horizon_gmv,6.070012,seasonal_correction
219,ly_calendar_horizon_gmv,4.952295,seasonal_correction
221,is_lapsed_vs_last_year,2.747453,seasonal_correction
222,ly_calendar_avg_order_value,2.545786,seasonal_correction
223,ly_pre_14d_gmv,2.453014,seasonal_correction
224,ly_calendar_gmv_days,2.335677,seasonal_correction
226,ly_calendar_horizon_to_ord,1.971204,seasonal_correction
231,ly_weekday_horizon_to_ord,1.377136,seasonal_correction
232,scaled_ly_weekday_gmv,1.311299,seasonal_correction
235,yoy_gmv_scale_14d,1.257256,seasonal_correction


## 9. Решение

Сезонный корректор принимается только при положительном incremental RMSLE относительно current correction и выигрыше на большинстве user-fold. Даже при принятии результат остаётся ограниченным одним зимним якорем, поэтому финальное применение должно быть вынесено в отдельный submission-ноутбук.

In [10]:
seasonal_fold_wins = int((fold_pivot.seasonal_improvement_vs_current > 0).sum())
accepted = bool(
    seasonal_score < base_score
    and seasonal_score < current_score
    and seasonal_fold_wins >= 3
)
summary = {
    'validation_anchor': VALIDATION_ANCHOR,
    'n_users': len(dataset),
    'n_current_features': len(control_feature_columns),
    'n_seasonal_features': len(seasonal_features),
    'correction_limit_log1p': CORRECTION_LIMIT,
    'base_rmsle': base_score,
    'constant_correction_rmsle': float(comparison.loc[comparison.model == 'constant_correction', 'rmsle'].iloc[0]),
    'current_correction_rmsle': current_score,
    'seasonal_correction_rmsle': seasonal_score,
    'seasonal_improvement_vs_current': seasonal_increment,
    'seasonal_fold_wins': seasonal_fold_wins,
    'accepted_for_submission_experiment': accepted,
    'validation_limitation': 'one winter anchor; user-fold validation does not test transfer across time',
}
with open(CATBOOST_CORRECTION_DIR / 'summary.json', 'w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)
display(pd.DataFrame([summary]))

,validation_anchor,n_users,n_current_features,n_seasonal_features,correction_limit_log1p,base_rmsle,constant_correction_rmsle,current_correction_rmsle,seasonal_correction_rmsle,seasonal_improvement_vs_current,seasonal_fold_wins,accepted_for_submission_experiment,validation_limitation
0,2026-01-14,250000,217,59,0.2,1.703598,1.684658,1.684343,1.682182,0.00216,5,True,one winter anchor; user-fold validation does n...


## 10. Итог эксперимента

На зимнем якоре `2026-01-14` базовая модель CatBoost получила RMSLE **1.703598**. Постоянная поправка в `log1p` снизила его до **1.684658**, корректор только по текущим признакам — до **1.684343**, а корректор с year-over-year признаками — до **1.682182**.

Чистый выигрыш от добавления сезонности к тому же корректору равен **0.002160 RMSLE**. Он положителен на всех 5 user-fold: от **0.001935** до **0.002433**. Наиболее важными сезонными признаками стали GMV в прогнозном окне год назад, его календарный аналог, флаг ушедшего пользователя и его прошлогодний средний чек.

Сезонный корректор **принимается как кандидат для submission-эксперимента**, но пока не как безусловно лучшая финальная схема. Причина: cross-user валидация проверяет новых пользователей внутри одной даты, но не доказывает перенос общей калибровки с `2026-01-14` на финальный якорь `2026-02-13`. Кроме того, 63–71% предсказанных поправок упираются в ограничение `-0.2`, что показывает сильное общее завышение базового прогноза на этом якоре. Следующий шаг — отдельный ноутбук с финальным обучением и двумя submission: базовым и сезонно скорректированным.